#### Pydantic

Pydantic 模型提供自动验证

In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from agent_lab.model_hub import LLM_FAST

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")

model = init_chat_model(**LLM_FAST)

In [2]:
model_with_structure = model.with_structured_output(Movie)
response1 = model_with_structure.invoke("Provide details about the movie Inception")

In [3]:
response1

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

获取解析后的输出和原始 AI 消息:

In [4]:
model_with_raw_structure = model.with_structured_output(Movie, include_raw=True)
response2 = model_with_raw_structure.invoke("Provide details about the movie Inception")

In [5]:
response2

{'raw': AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 365, 'total_tokens': 454, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 109}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': 'de6a1c6f-c0c9-49fa-b7d3-fad9b927309c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dc9a7-e61c-79d1-920a-b561dc62ba2e-0', tool_calls=[{'name': 'Movie', 'args': {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}, 'id': 'call_00_0IWwIp1CL7yaHAjecEW3DYKO', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 365, 'output_tokens': 89, 'total_tokens': 454, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}),
 'parsed':

In [6]:
response2['raw'].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  Movie (call_00_0IWwIp1CL7yaHAjecEW3DYKO)
 Call ID: call_00_0IWwIp1CL7yaHAjecEW3DYKO
  Args:
    title: Inception
    year: 2010
    director: Christopher Nolan
    rating: 8.8


#### TypedDict

In [7]:
from typing import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [8]:
model_with_structure3 = model.with_structured_output(MovieDict)
response3 = model_with_structure3.invoke("Provide details about the movie Inception")

In [9]:
response3

{'title': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 8.8}

#### JSON Schema

method 参数：

- `json_schema` ：使用提供商提供的专用结构化输出功能。

- `function_calling` ：通过强制按给定模式调用工具来导出结构化输出。

In [10]:
json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

In [ ]:
model_with_structure4 = model.with_structured_output(
    json_schema,
    method="json_schema",
)
response4 = model_with_structure4.invoke("Provide details about the movie Inception")

In [12]:
response4

{'title': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 8.8}